In [ ]:
mssparkutils.notebook.run("config", 60)

In [ ]:
empresas = [
    {
    "nome": "empresa 01", 
    "app_key": spark.conf.get("empresa_01_app_key"),
    "app_secret": spark.conf.get("empresa_01_app_secret"),
    "contas": [2933348613, 3058198316]
    },
    {
    "nome": "empresa 02",
    "app_key": spark.conf.get("empresa_02_app_key"),
    "app_secret": spark.conf.get("empresa_02_app_secret"),
    "contas": [10947358827, 10947358922, 10947359034]
    },
    {
    "nome": "empresa 03",
    "app_key": spark.conf.get("empresa_03_app_key"),
    "app_secret": spark.conf.get("empresa_03_app_secret"),
    "contas": [10435241406, 10488799809]
    }
]

In [ ]:
from datetime import datetime, timedelta

url = "https://app.omie.com.br/api/v1/financas/extrato/"
call = "ListarExtrato"
data_key = "listaMovimentos"
nome_tabela = "extrato_bancario"
data_inicio = datetime(2025, 1, 1)
data_fim = datetime.today()

In [ ]:
def gerar_meses(data_inicio, data_fim):
    meses = []
    atual = data_inicio.replace(day=1)

    while atual <= data_fim:
        meses.append(atual)
        if atual.month == 12:
            atual = atual.replace(year=atual.year + 1, month=1)
        else:
            atual = atual.replace(month=atual.month + 1)

    return meses

In [ ]:
import requests

def get_dados(call, app_key, app_secret, conta, data_inicio, data_fim, data_key):

    pagina = 1
    todos_dados 
    
    payload = { 
         "call": call,
         "app_key": app_key,
         "app_secret": app_secret,
         "param": [{
                     "nCodCC": conta,
                     "dPeriodoInicial": data_inicio,
                    "dPeriodoFinal": data_fim
            }]
        }

    response = requests.post(url, json=payload)

    if response.status_code != 200:
            raise Exception(f"Erro na API: {response.text}")

    data = response.json()

    return data.get(data_key, [])

In [ ]:
meses = gerar_meses(data_inicio, data_fim)

todos_dados = []

for emp in empresas: 
    print(f"Processando {emp['nome']}...")

    for conta in emp["contas"]:
        print(f"   Conta: {conta}")

        for mes in meses:

            inicio = mes.replace(day=1)

            if mes.month == 12:
                fim = mes.replace(day=31)
            else:
                fim = (mes.replace(month=mes.month + 1, day=1) - timedelta(days=1))

            print(f"      -> Período: {inicio.strftime('%d/%m/%Y')} até {fim.strftime('%d/%m/%Y')}")

            dados = get_dados(
                call,
                emp["app_key"],
                emp["app_secret"],
                conta,
                inicio.strftime("%d/%m/%Y"),
                fim.strftime("%d/%m/%Y"),
                data_key
            )

            for d in dados:
                d["empresa"] = emp["nome"]
                d["conta"] = conta

            todos_dados.extend(dados)

In [ ]:
print(len(todos_dados))

In [ ]:
import json

todos_dados_json = [(json.dumps(d),) for d in todos_dados]

df_bronze = spark.createDataFrame(todos_dados_json, ["raw_json"])

In [ ]:
display(df_bronze)

In [ ]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp

df_bronze = df_bronze.withColumn("ingestion_ts", from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))

display(df_bronze)

In [ ]:
caminho = f"Files/bronze/{nome_tabela}"
df_bronze.write.format("delta").mode("append").save(caminho)
print(f"Tabela {nome_tabela} carregada com sucesso")